#MECE

| Member    | Task                            | Description                                                                                   |
| ------------- | ----------------------------------- | ------------------------------------------------------------------------------------------------- |
| Devanshi Adhikari  | Data Preprocessing                  | Download Yelp dataset, extract image-label pairs, resize/normalize images, split train/test sets. |
| Anirudhra Budathoki | kNN Model Development               | Implement k-Nearest Neighbors classifier using image features or embeddings. Evaluate kNN using confusion matrix and AUC; interpret results                      |
| Arjun Kumar    | kNN Tuning                    | Tuning of KNN model                                  |
| Sumith  Padma Padmasree Vijayapadman  | SVM Model Development               | Train SVM on image features (e.g., PCA or embeddings). No tuning required.                        |
| Jeffin John Abraham     | SVM Evaluation                      | Compute confusion matrix and AUC for SVM; analyze performance.                                    |
| Stephen David Chitilapalli Mathew  | DNN Implementation                  | Build and train a deep neural network (fully connected layers) for image classification.          |
| Shabda   Kafle | DNN Evaluation and Tuning           | Evaluate DNN with confusion matrix and AUC; perform hyperparameter tuning.                        |
| Aparna Jayakumar Resmi   | CNN Implementation                  | Design and train a convolutional neural network for image classification.                         |
| Neha  Tamang    | CNN Evaluation and Interpretability | Evaluate CNN using confusion matrix and AUC; apply interpretability techniques.|

# Data Preprocessing

In [ ]:
import os
import pandas as pd
import shutil
import random
from PIL import Image
from tqdm import tqdm
photos_dir = "/content/drive/MyDrive/yelp/photos"
csv_path = "/content/drive/MyDrive/yelp/photos.csv"
output_dir = "/content/yelp_processed"
os.makedirs(output_dir, exist_ok=True)
IMG_SIZE = (224, 224)


In [ ]:
df = pd.read_csv(csv_path)
df = df.dropna(subset=['label'])
df['filename'] = df['photo_id'] + ".jpg"
df = df[df['filename'].apply(lambda x: os.path.isfile(os.path.join(photos_dir, x)))]
print(f"Total valid images found: {len(df)}")


In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df['split'] = 'train'
test_df['split'] = 'test'
full_df = pd.concat([train_df, test_df])


In [ ]:
import concurrent.futures
success_count = 0
failure_count = 0
for split in ['train', 'test']:
    for label in df['label'].unique():
        os.makedirs(os.path.join(output_dir, split, label), exist_ok=True)
def process_and_copy_threaded(row):
    global success_count, failure_count
    src_path = os.path.join(photos_dir, row['filename'])
    dst_path = os.path.join(output_dir, row['split'], row['label'], row['filename'])
    try:
        img = Image.open(src_path).convert("RGB")
        img = img.resize(IMG_SIZE)
        img.save(dst_path)
        return True
    except Exception as e:
        print(f"Skipping {row['filename']} — Error: {str(e)}", flush=True)
        return False
rows = [row for _, row in full_df.iterrows()]
success_count = 0
failure_count = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    results = list(tqdm(executor.map(process_and_copy_threaded, rows), total=len(rows)))
success_count = sum(results)
failure_count = len(results) - success_count

print(f"\nFinished processing.")
print(f"Successful: {success_count}")
print(f"Failed: {failure_count}")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/yelp_processed/train"
test_dir = "/content/yelp_processed/test"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
# Normalize pixel values (0-1)
datagen = ImageDataGenerator(rescale=1./255)
train_data = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


In [ ]:
import shutil
shutil.make_archive('/content/yelp_processed', 'zip', '/content/yelp_processed')


In [ ]:
destination = '/content/drive/MyDrive/yelp_processed.zip'
shutil.copy('/content/yelp_processed.zip', destination)


### Data Processing 2

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from skimage import exposure
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Parameters
IMG_SIZE = 128
IMAGE_DIR = 'photos/'
MAX_IMAGES = 50000

# Load metadata
photos = pd.read_json("photos.json", lines=True)
photos = photos[photos['label'].notna()][['photo_id', 'label']]
le = LabelEncoder()
photos['label'] = le.fit_transform(photos['label'])

# Preprocessing function
def advanced_preprocess(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    img[binary == 0] = 0

    for i in range(3):
        img[:, :, i] = exposure.equalize_hist(img[:, :, i]) * 255

    img = cv2.GaussianBlur(img, (3, 3), 0)
    return img.astype(np.float32) / 255.0

# Process images
images, labels, skipped = [], [], 0
for i, row in enumerate(photos.itertuples(index=False)):
    if i >= MAX_IMAGES:
        break
    path = os.path.join(IMAGE_DIR, f"{row.photo_id}.jpg")
    try:
        img_array = advanced_preprocess(path)
        images.append(img_array)
        labels.append(row.label)
    except:
        skipped += 1
print(f"Finished processing. Skipped: {skipped}")

# Save arrays
X = np.array(images, dtype=np.float32)
y = np.array(labels)
np.save("X_images.npy", X)
np.save("y_labels.npy", y)

# Model Building - CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

# Dataset class that loads images and labels lazily using memmap and indices
class MemmapDataset(Dataset):
    def __init__(self, image_path, label_path, indices):
        self.X = np.load(image_path, mmap_mode='r')
        self.y = np.load(label_path, mmap_mode='r')
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.X[i].transpose(2, 0, 1)  # Convert HWC to CHW
        y = self.y[i]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Load labels to get number of classes and for stratified splits
label_path = '/content/drive/MyDrive/Comp_Vis/y_labels.npy'
image_path = '/content/drive/MyDrive/Comp_Vis/X_images.npy'

all_labels = np.load(label_path, mmap_mode='r')
num_classes = len(np.unique(all_labels))

# Create train/val/test indices with stratification
full_indices = np.arange(len(all_labels))
train_idx, test_idx = train_test_split(full_indices, test_size=0.2, stratify=all_labels, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, stratify=all_labels[train_idx], random_state=42)

# Create datasets and dataloaders
train_dataset = MemmapDataset(image_path, label_path, train_idx)
val_dataset = MemmapDataset(image_path, label_path, val_idx)
test_dataset = MemmapDataset(image_path, label_path, test_idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CNN model definition
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

# Evaluation function (also returns loss if requested)
def evaluate_model(model, data_loader, return_loss=False):
    model.eval()
    preds, probs, labels = [], [], []
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0
    with torch.no_grad():
        for Xb, yb in data_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            output = model(Xb)
            loss = loss_fn(output, yb)
            total_loss += loss.item()
            prob = torch.softmax(output, dim=1).cpu().numpy()
            pred = np.argmax(prob, axis=1)
            preds.extend(pred)
            probs.extend(prob)
            labels.extend(yb.cpu().numpy())

    if return_loss:
        return total_loss / len(data_loader)

    # Confusion matrix plot
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("CNN Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # AUC score
    y_true_bin = label_binarize(labels, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, np.array(probs), multi_class='ovr')
    print(f"AUC Score: {auc:.4f}")

# Training function with progress bar and early stopping
def train_model(model, train_loader, val_loader, epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)
    criterion = nn.CrossEntropyLoss()
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        print(f"\nEpoch {epoch+1}/{epochs}")
        for Xb, yb in tqdm(train_loader, desc="Training", leave=False):
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            output = model(Xb)
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_loss = evaluate_model(model, val_loader, return_loss=True)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} - Train Loss: {total_loss/len(train_loader):.4f} - Val Loss: {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 5:
            print("Early stopping triggered.")
            break

# Initialize and train model
model = ImprovedCNN(num_classes).to(device)
train_model(model, train_loader, val_loader)

# Final evaluation on test set
print("\nFinal Evaluation on Test Set")
evaluate_model(model, test_loader)

AUC Score: 0.9246

## CNN Evaluation and Tuning

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import label_binarize

In [ ]:
# Load data
X = np.load("/content/drive/MyDrive/X_images.npy", mmap_mode='r')[:10000]
y = np.load('/content/drive/MyDrive/y_labels.npy')[:10000]
num_classes = len(np.unique(y))

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, stratify=y, random_state=42
 )


In [ ]:
# PyTorch Tensors: (N, C, H, W)
X_train = torch.tensor(X_train.transpose(0, 3, 1, 2), dtype=torch.float32)
X_test = torch.tensor(X_test.transpose(0, 3, 1, 2), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

In [ ]:
# CNN Model
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x)

In [ ]:
# Evaluate
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('CNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# AUC Score (Multiclass One-vs-Rest)
y_true_bin = label_binarize(all_labels, classes=np.arange(num_classes))
y_probs = np.array(all_probs)

auc_score = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
print(f"CNN Multiclass AUC Score: {auc_score:.4f}")

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=10, device='cpu'):
    model.to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

    print("Finished Training")

In [ ]:
# Evaluate the trained model
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
# Evaluate the trained model
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

auc_score = roc_auc_score(label_binarize(all_labels, classes=np.arange(num_classes)), np.array(all_probs), multi_class='ovr')
print(f"CNN Multiclass AUC Score: {auc_score:.4f}")

In [ ]:
#tuning different learning rates and batch sizes
learning_rates = [0.01, 0.001, 0.0001]
batch_sizes = [32, 64, 128]
results = {}

device = "cuda" if torch.cuda.is_available() else "cpu"

for lr in learning_rates:
    for bs in batch_sizes:
        print(f"Training with Learning Rate: {lr}, Batch Size: {bs}")
        # Create new DataLoaders with the current batch size
        train_loader_tuned = DataLoader(TensorDataset(X_train, y_train), batch_size=bs, shuffle=True)
        test_loader_tuned = DataLoader(TensorDataset(X_test, y_test), batch_size=bs)

        model_tuned = CNNModel(num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer_tuned = optim.Adam(model_tuned.parameters(), lr=lr)

In [ ]:
 # The train_model function now returns cm and auc_score
cm_tuned, auc_score_tuned = train_model(model_tuned, train_loader_tuned, test_loader_tuned, criterion, optimizer_tuned, num_epochs=5, device=device)

results[(lr, bs)] = {'confusion_matrix': cm_tuned, 'auc_score': auc_score_tuned}
print(f"  AUC Score: {auc_score_tuned:.4f}")


## CNN Interpretability

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import torch

# Select a sample index from test dataset
sample_idx = 500

# Since your MemmapDataset returns (C,H,W) tensors, convert to (H,W,C) numpy image for LIME
img_tensor, true_label = test_dataset[sample_idx]
img_np = img_tensor.permute(1, 2, 0).numpy()

# Define prediction function for LIME
def batch_predict(images):
    # images: list or numpy array of images (N, H, W, C) with pixel values 0-255 usually
    model.eval()
    batch = torch.tensor(images).permute(0, 3, 1, 2).float().to(device)  # NHWC -> NCHW
    with torch.no_grad():
        logits = model(batch)
        probs = torch.softmax(logits, dim=1)
    return probs.cpu().numpy()

# Initialize LIME explainer
explainer = lime_image.LimeImageExplainer()

# Run explanation on the sample image
explanation = explainer.explain_instance(
    img_np.astype(np.double),    # lime expects double precision float
    classifier_fn=batch_predict,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

# Get mask and image with boundaries for the top predicted label
temp, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=True,
    hide_rest=False,
    num_features=10,
    min_weight=0.0
)

# Plot the explanation
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title(f"LIME Interpretation (True label: {true_label.item()}, Predicted: {explanation.top_labels[0]})")
plt.axis('off')
plt.show()

# Model Building - SVM

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

# Load preprocessed data
X = np.load('X_images.npy')
y = np.load('y_labels.npy')
num_classes = len(np.unique(y))

# Sample 10000 images
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=10000, stratify=y, random_state=42)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, stratify=y_sample, random_state=42)

# Flatten images (N, C, H, W) -> (N, C*H*W)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Optional: PCA for dimensionality reduction
pca = PCA(n_components=0.95)  # Keep 95% of variance
X_train_pca = pca.fit_transform(X_train_flat)
X_test_pca = pca.transform(X_test_flat)

# Train SVM Model
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_pca, y_train)

# Predictions and probability outputs
y_preds = svm_model.predict(X_test_pca)
y_probs = svm_model.predict_proba(X_test_pca)

# Confusion Matrix
cm = confusion_matrix(y_test, y_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# AUC Score (Multiclass One-vs-Rest)
y_true_bin = label_binarize(y_test, classes=np.arange(num_classes))
auc_score = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
print(f"SVM Multiclass AUC Score: {auc_score:.4f}")

SVM Multiclass AUC Score: 0.8687

## SVM Model Evaluation


In [ ]:
import pandas as pd

image_extract_path = '/content/yelp_images'

json_path = os.path.join(image_extract_path, 'photos.json')
df = pd.read_json(json_path, lines=True)

target_labels = ['food', 'drink', 'menu', 'inside']
df_filtered = df[df['label'].isin(target_labels)]

df_sampled = df_filtered.groupby('label').apply(lambda x: x.sample(n=1250, random_state=42)).reset_index(drop=True)

print("Total images selected:", len(df_sampled))


In [ ]:
import numpy as np
import cv2
from tqdm import tqdm

IMG_SIZE = 64
image_dir = os.path.join(image_extract_path, 'photos')
X = []
y = []

for idx, row in tqdm(df_sampled.iterrows(), total=len(df_sampled)):
    img_id = row['photo_id']
    label = row['label']
    img_path = os.path.join(image_dir, img_id + '.jpg')

    try:
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        X.append(img.flatten())
        y.append(label)
    except:
        continue

X = np.array(X)
y = np.array(y)

print("Loaded images:", X.shape[0])
print("Feature shape per image:", X.shape[1])


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=200, random_state=42)
X_pca = pca.fit_transform(X)
print("PCA-reduced shape:", X_pca.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_pca, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
print("SVM model training completed.")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = svm_model.predict(X_test)
y_prob = svm_model.predict_proba(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_test, y_pred, target_names=le.classes_))

y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
print("Macro-Averaged AUC Score:", round(auc, 4))


# Model Building - DNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Custom Dataset for DNN that flattens image
class MemmapDNNFlatDataset(Dataset):
    def __init__(self, image_path, label_path, indices):
        self.X = np.load(image_path, mmap_mode='r')
        self.y = np.load(label_path, mmap_mode='r')
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.X[i].transpose(2, 0, 1).reshape(-1)  # HWC → CHW → flat
        y = self.y[i]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Paths
image_path = '/content/drive/MyDrive/X_images.npy'
label_path = '/content/drive/MyDrive/y_labels.npy'

# Load labels and split
all_labels = np.load(label_path, mmap_mode='r')
num_classes = len(np.unique(all_labels))

indices = np.arange(len(all_labels))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=all_labels, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, stratify=all_labels[train_idx], random_state=42)

# Datasets and Loaders
train_dataset = MemmapDNNFlatDataset(image_path, label_path, train_idx)
val_dataset = MemmapDNNFlatDataset(image_path, label_path, val_idx)
test_dataset = MemmapDNNFlatDataset(image_path, label_path, test_idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

# DNN Model
class DNNModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Evaluation
def evaluate_model(model, data_loader, return_loss=False):
    model.eval()
    preds, probs, labels = [], [], []
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    with torch.no_grad():
        for Xb, yb in data_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            loss = criterion(out, yb)
            total_loss += loss.item()
            prob = torch.softmax(out, dim=1).cpu().numpy()
            pred = np.argmax(prob, axis=1)
            preds.extend(pred)
            probs.extend(prob)
            labels.extend(yb.cpu().numpy())

    if return_loss:
        return total_loss / len(data_loader)

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.title("DNN Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    y_true_bin = label_binarize(labels, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, np.array(probs), multi_class='ovr')
    print(f"DNN AUC Score: {auc:.4f}")

# Training
def train_model(model, train_loader, val_loader, epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        print(f"\nEpoch {epoch+1}/{epochs}")
        for Xb, yb in tqdm(train_loader, desc="Training", leave=False):
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_loss = evaluate_model(model, val_loader, return_loss=True)
        scheduler.step(val_loss)
        print(f"Train Loss: {total_loss/len(train_loader):.4f} - Val Loss: {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 5:
            print("Early stopping triggered.")
            break

# Run training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_input, _ = train_dataset[0]
input_dim = sample_input.shape[0]

model = DNNModel(input_dim=input_dim, num_classes=num_classes).to(device)
train_model(model, train_loader, val_loader)

# Final Test Evaluation
print("\nFinal Test Evaluation")
evaluate_model(model, test_loader)

## DNN Evaluation and Tuning

In [ ]:
#Evaluation

from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(torch.argmax(probs, dim=1).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

In [ ]:
# Stack all batches
y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Classification Report
print("\nClassification Report:\n", classification_report(y_true, y_pred))


In [ ]:
# AUC Score (One-vs-Rest)
try:
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
    print(f"\nAUC Score (One-vs-Rest): {auc:.4f}")
except Exception as e:
    print(f"\nAUC could not be computed: {e}")

In [ ]:
#tuning 1: changing the epoch to 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train_dnn.shape[1]
model = DNNModel(input_dim=input_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[DNN] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
#Evaluation 1

from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(torch.argmax(probs, dim=1).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

# Stack all batches
y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)

In [ ]:
# Confusion matrix 1
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Classification Report 1
print("\nClassification Report:\n", classification_report(y_true, y_pred))

In [ ]:
# AUC Score (One-vs-Rest) 1
try:
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
    print(f"\nAUC Score (One-vs-Rest): {auc:.4f}")
except Exception as e:
    print(f"\nAUC could not be computed: {e}")

In [ ]:
#tuning 2: changing the epoch to 10 and adjusting the dropout rate

# DNN MODEL
class tunedDNNModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(tunedDNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train_dnn.shape[1]
model = tunedDNNModel(input_dim=input_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[DNN] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
#Evaluation 2

from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(torch.argmax(probs, dim=1).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

# Stack all batches
y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)

In [ ]:
# Confusion matrix 2
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Classification Report 2
print("\nClassification Report:\n", classification_report(y_true, y_pred))

In [ ]:
# AUC Score (One-vs-Rest) 2
try:
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
    print(f"\nAUC Score (One-vs-Rest): {auc:.4f}")
except Exception as e:
    print(f"\nAUC could not be computed: {e}")

In [ ]:
#tuning 3: adding the extra layer and changing the epoch to 10
class TunedDNNModel(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rate=0.3):
        super(TunedDNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train_dnn.shape[1]
model = TunedDNNModel(input_dim=input_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[DNN] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss / len(train_loader):.4f}")



In [ ]:
#Evaluation 3

from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(torch.argmax(probs, dim=1).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

# Stack all batches
y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)

In [ ]:
# Confusion matrix 3
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Classification Report 3
print("\nClassification Report:\n", classification_report(y_true, y_pred))

In [ ]:
# AUC Score (One-vs-Rest) 3
try:
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    auc = roc_auc_score(y_true_bin, y_probs, multi_class='ovr')
    print(f"\nAUC Score (One-vs-Rest): {auc:.4f}")
except Exception as e:
    print(f"\nAUC could not be computed: {e}")

# Model Building - KNN

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
#SET PATHS
photos_dir = 'photos'  # Directory containing photo files
photos_json = 'photos.json'  # Path to JSON file
output_dir = 'preprocessed_data'  # Where to save processed data
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Constants
IMG_SIZE = 128  # For CNN
MAX_IMAGES = 50000

# Load JSONL file (each line is a JSON object)
photos_data = []
with open(photos_json, 'r') as f:
    for line in f:
        try:
            photos_data.append(json.loads(line.strip()))
        except json.JSONDecodeError as e:
            print(f'Error parsing line: {e}')
            continue

# Convert to DataFrame
photos_df = pd.DataFrame(photos_data)
photos_df = photos_df.sample(n=20000, random_state=42).reset_index(drop=True)

print('Sampled DataFrame shape:', photos_df.shape)
print(photos_df.head())


In [ ]:
# Label distribution
print("\nLabel distribution:\n", photos_df['label'].value_counts())

# Check missing values and caption lengths
print("\nMissing captions:", photos_df['caption'].isnull().sum())
print("\nCaption length stats:\n", photos_df['caption'].str.len().describe())

In [ ]:
# Plot label distribution
import matplotlib.pyplot as plt

label_counts = photos_df['label'].value_counts()
plt.figure(figsize=(8, 6))
label_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Labels')
plt.xlabel('Label')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Caption length boxplot
photos_df['caption_length'] = photos_df['caption'].str.len()
plt.figure(figsize=(10, 6))
photos_df.boxplot(column='caption_length', by='label', grid=False, showfliers=False, patch_artist=True, medianprops={'color': 'black'})
plt.title('Caption Length Distribution by Label')
plt.suptitle('')
plt.xlabel('Label')
plt.ylabel('Caption Length')
plt.show()

In [ ]:
# Unique businesses by label
business_by_label = photos_df.groupby('label')['business_id'].nunique()
print("\nUnique businesses per label:\n", business_by_label)

In [ ]:
# Encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
photos_df['label'] = le.fit_transform(photos_df['label'])

In [ ]:
# Process images
images = []
labels = []
skipped_count = 0
total = min(MAX_IMAGES, len(photos_df))

print(f"\nTotal images to process: {total}")

for i, row in enumerate(photos_df.itertuples(index=False)):
    if i >= MAX_IMAGES:
        break

    img_path = os.path.join(photos_dir, f"{row.photo_id}.jpg")  #  use photo_id

    try:
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
            img_array = np.asarray(img, dtype=np.float32) / 255.0  # Normalize
            images.append(img_array)
            labels.append(row.label)
    except Exception as e:
        skipped_count += 1
        print(f"Failed to process {img_path}: {e}")

    if i % (total // 20 or 1) == 0 or i == total - 1:
        percent = (i + 1) / total * 100
        print(f"Progress: {percent:.1f}% ({i + 1}/{total})", end='\r')

print(f"\n\nFinished processing {len(images)} images.")
print(f"Skipped {skipped_count} corrupted/unreadable images.")

In [ ]:
test_path = os.path.join(photos_dir, f"{photos_df.iloc[0].photo_id}.jpg")
print("Test path:", test_path)

try:
    with Image.open(test_path) as img:
        img = img.convert('RGB')
        img = img.resize((64, 64))
        img_array = np.asarray(img, dtype=np.float32) / 255.0
        print("Image shape:", img_array.shape)
except Exception as e:
    print("Error:", e)


In [ ]:
# 1. Flatten image arrays
X = np.array(images).reshape(len(images), -1)  # (num_samples, 64*64*3)
y = np.array(labels)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV


In [ ]:
#create sample data if you're just testing
import numpy as np
X = np.random.rand(100, 4)  # Create 100 samples with 4 features
y = np.random.randint(0, 2, 100)  # Binary target variable

# Now split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 3. Train KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)


In [ ]:
# 4. Predict and evaluate
y_pred = knn.predict(X_test)


In [ ]:
distances, indices = knn.kneighbors([X_test[0]])
print("Nearest labels:", y_train[indices])


## KNN Evaluation

In [ ]:
print("\n--- Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
images = []
labels = []

for i, row in enumerate(photos_df.itertuples(index=False)):
    img_path = os.path.join(photos_dir, f"{row.photo_id}.jpg")

    try:
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            img = img.resize((224, 224))  # required size for ResNet
            img_array = img_to_array(img)
            img_array = preprocess_input(img_array)
            images.append(img_array)
            labels.append(row.label)
    except Exception as e:
        continue

X = np.array(images)
y = np.array(labels)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt  # Also adding matplotlib import to be safe

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Import the required functions from scikit-learn
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt  # Also adding this import in case it's missing

# y_pred_proba gives probability for both classes. Take [:,1] for positive class
y_score = knn.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_score)
print("AUC Score:", auc_score)

# Optional: plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_score)
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.2f}')
plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score
import numpy as np

# Step 1: Convert labels to binary matrix
classes = np.unique(y_test)

# Check if binary or multi-class classification
if len(classes) == 2:
    # For binary classification, use the raw probabilities of the positive class
    y_score = knn.predict_proba(X_test)[:, 1]  # Use probability of positive class only
    auc_score = roc_auc_score(y_test, y_score)
    print("AUC (binary):", auc_score)
else:
    # For multi-class, use label_binarize
    y_test_bin = label_binarize(y_test, classes=classes)
    y_score = knn.predict_proba(X_test)

    # Step 2: Calculate macro-average AUC
    auc_score = roc_auc_score(y_test_bin, y_score, average="macro", multi_class="ovr")
    print("Macro-Averaged AUC (OvR):", auc_score)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier


# KNN Tuning

In [ ]:
# Define your parameter grid
param_grid = {
    'n_neighbors': list(range(1, 21)),          # Try 1 to 20 neighbors
    'weights': ['uniform', 'distance'],         # Weight strategies
    'metric': ['euclidean', 'manhattan']        # Distance metrics
}

# Initialize model
knn = KNeighborsClassifier()

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

# Fit the search
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy Score:", grid_search.best_score_)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

k_range = range(1, 21)
accuracies = []

for k in k_range:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

plt.plot(k_range, accuracies, marker='o')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. k for KNN')
plt.grid(True)
plt.show()


# Prompts used:

- First prompt: The data has two components a folder names photos and a file names photos.json. Perform data preprocessing and build CNN model on it.

 - Last prompt:     replace this with something to use less RAM and donot use sample dataset.
 ```
X = np.load('/content/drive/MyDrive/Comp_Vis/X_images.npy')
y = np.load('/content/drive/MyDrive/Comp_Vis/y_labels.npy')
```

- first prompt- if someone built a dnn model, how can i go about evaluating it with confusion matrix and AUC.
 - last prompt- what else can be done to tune a model

- how do i make confusion matrix and AUC for CNN model.
 can you guide me to tune the model too

- NN Model build up: ROC paramter tuning and KNN implementation
- Evaluation of KNN model with AUC,ROC as classification reports
- First prompt : how to convert json file to csv.
  - Last prompt : multithreading in test and train
- How to build a dnn model for the given dataset
- Sample the dataset and build SVM model with them
  